In [1]:
from pyspark.sql import Window
from pyspark.sql.functions import col, avg, lag, to_date

try:
    if not spark.catalog.tableExists("silver_nasdaq_stocks"):
        raise ValueError("Silver table missing! Run the Silver pipeline first.")

    df_silver = spark.read.table("silver_nasdaq_stocks")

    # 1. Define a window partitioned by Company and ordered by Date for time-series calculations
    company_window = Window.partitionBy("Company").orderBy("Date")

    # 2. Compute advanced financial data engineering metrics
    df_gold_enriched = df_silver.withColumn("Prev_Close", lag("Close_Price", 1).over(company_window)).withColumn("Daily_Return_Pct", ((col("Close_Price") - col("Prev_Close")) / col("Prev_Close")) * 100).withColumn("MA_30_Close", avg("Close_Price").over(company_window.rowsBetween(-29, 0))).withColumn("Volume_MA_30", avg("Volume").over(company_window.rowsBetween(-29, 0)))

    # 3. Write out to the Gold Fact Table
    df_gold_enriched.write.format("delta").mode("overwrite").saveAsTable("gold_fact_nasdaq_trades")

    # 4. Optimize the Gold table storage
    spark.sql("OPTIMIZE gold_fact_nasdaq_trades ZORDER BY (Company, Date)")
    
    print("Gold Layer: Fact table with rolling financial metrics successfully created and optimized!")

except Exception as e:
    print(f"Gold Transformation Pipeline Failed: {str(e)}")
    raise e

StatementMeta(, f2c64377-64ef-4940-82a9-1896579d1b10, 3, Finished, Available, Finished, False)

Gold Layer: Fact table with rolling financial metrics successfully created and optimized!
